# Práctica N.° 04 — Arreglos dinámicos, vectores y matrices
**Curso:** Algoritmos y Estructuras de Datos (SIS210) · UNAP 2026

Actividades 1 y 4 (Python).

## Actividad 1 — Acceso O(1) vs. inserción al inicio O(n)

In [ ]:
# =============================================================
# Practica N.o 04 - ACTIVIDAD 1 (Python)
# Acceso O(1) vs insercion al inicio O(n)
# =============================================================
import time
import statistics
from collections import deque
import matplotlib.pyplot as plt

ns = [1_000, 5_000, 10_000, 50_000, 100_000]
R = 5                                   # repeticiones por tamano
tiempos_append, tiempos_insert0, tiempos_deque = [], [], []

for n in ns:
    ta, ti, td = [], [], []
    for _ in range(R):
        # --- append: O(1) amortizado ---
        lst = []
        t0 = time.perf_counter()
        for i in range(n):
            lst.append(i)
        ta.append((time.perf_counter() - t0) * 1000)

        # --- insert(0, x): O(n) por insercion ---
        lst = []
        t0 = time.perf_counter()
        for i in range(n):
            lst.insert(0, i)
        ti.append((time.perf_counter() - t0) * 1000)

        # --- deque.appendleft: O(1) (para la pregunta 8.4) ---
        d = deque()
        t0 = time.perf_counter()
        for i in range(n):
            d.appendleft(i)
        td.append((time.perf_counter() - t0) * 1000)

    tiempos_append.append(statistics.median(ta))
    tiempos_insert0.append(statistics.median(ti))
    tiempos_deque.append(statistics.median(td))

print(f"{'n':>10} {'append (ms)':>15} {'insert(0) (ms)':>16} {'ratio':>8} {'deque (ms)':>12}")
for n, tapp, tins, tdq in zip(ns, tiempos_append, tiempos_insert0, tiempos_deque):
    print(f"{n:>10} {tapp:>15.3f} {tins:>16.3f} {tins/tapp:>7.1f}x {tdq:>12.3f}")

# --- Verificacion del acceso O(1): lst[i] en inicio, centro y final ---
print(f"\n{'n':>10} {'lst[0]':>10} {'lst[n//2]':>12} {'lst[n-1]':>11}   (ns por acceso)")
for n in ns:
    lst = list(range(n))
    tiempos = []
    for pos in (0, n // 2, n - 1):
        t0 = time.perf_counter_ns()
        for _ in range(100_000):
            x = lst[pos]
        tiempos.append((time.perf_counter_ns() - t0) / 100_000)
    print(f"{n:>10} {tiempos[0]:>10.1f} {tiempos[1]:>12.1f} {tiempos[2]:>11.1f}")

# --- Figura 1: escala log-log ---
plt.figure(figsize=(7, 3.6))
plt.loglog(ns, tiempos_insert0, 'o-', label='list.insert(0, x) - O(n) por operacion')
plt.loglog(ns, tiempos_append, 's-', label='list.append(x) - O(1) amortizado')
plt.loglog(ns, tiempos_deque, '^--', label='deque.appendleft(x) - O(1)')
plt.xlabel('n (numero de inserciones)')
plt.ylabel('Tiempo total (ms, escala log)')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Actividad 4 — NumPy vs. Python puro

In [ ]:
# =============================================================
# Practica N.o 04 - ACTIVIDAD 4 (Python)
# NumPy vs Python puro en multiplicacion de matrices
# =============================================================
import time
import statistics
import numpy as np


def multiplicar_matrices_puro(A, B):
    """Algoritmo clasico O(n^3) con listas de listas."""
    n = len(A)
    C = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C


def mediana_ms(funcion, repeticiones):
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        funcion()
        tiempos.append((time.perf_counter() - t0) * 1000)
    return statistics.median(tiempos)


np.random.seed(42)
print(f"{'n':>5} {'numpy (ms)':>12} {'puro (ms)':>12} {'ratio':>10} "
      f"{'GFLOP/s np':>12} {'GFLOP/s puro':>13}")

for n in [50, 100, 200]:
    A_np = np.random.rand(n, n)
    B_np = np.random.rand(n, n)
    A_py = A_np.tolist()                      # lista de listas Python pura
    B_py = B_np.tolist()

    ms_np = mediana_ms(lambda: A_np @ B_np, 50)
    ms_py = mediana_ms(lambda: multiplicar_matrices_puro(A_py, B_py), 3 if n < 200 else 1)

    # verificar que ambos metodos dan el mismo resultado
    C_py = np.array(multiplicar_matrices_puro(A_py, B_py))
    assert np.allclose(C_py, A_np @ B_np), "los resultados no coinciden"

    gflops_np = 2 * n ** 3 / (ms_np / 1000) / 1e9      # 2n^3 operaciones de punto flotante
    gflops_py = 2 * n ** 3 / (ms_py / 1000) / 1e9
    print(f"{n:>5} {ms_np:>12.3f} {ms_py:>12.2f} {ms_py/ms_np:>9.0f}x "
          f"{gflops_np:>12.1f} {gflops_py:>13.3f}")